In [ ]:
!pip install tensorflow=='2.15' --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorstore 0.1.67 requires ml-dtypes>=0.3.1, but you have ml-dtypes 0.2.0 which is incompatible.
tf-keras 2.17.0 requires tensorflow<2.18,>=2.17, but you have tensorflow 2.15.0 which is incompatible.


## Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import (Conv2D, BatchNormalization, ReLU, Add,
                                     GlobalAveragePooling2D, Dense, Input,
                                     Concatenate, Layer, Maximum, Dropout, Multiply)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import ImageFile

## Model Parameters

In [ ]:
IMG_HEIGHT = 256
IMG_WIDTH = 256
CHANNELS = 3
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, CHANNELS)
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)

BATCH_SIZE = 256
N_EPOCHS = 10
LR = 1e-3

VALUE_DT = 0.2
PATH_DIR = 'CelebA_Spoof_Faces/'

ImageFile.LOAD_TRUNCATED_IMAGES = True

## Model Functions

In [ ]:
class RGBtoHSV(Layer):
    def __init__(self, **kwargs):
        super(RGBtoHSV, self).__init__(**kwargs)

    def call(self, inputs):
        return tf.image.rgb_to_hsv(inputs)

    def get_config(self):
        config = super(RGBtoHSV, self).get_config()
        return config

class RGBtoYCbCr(Layer):
    def __init__(self, **kwargs):
        super(RGBtoYCbCr, self).__init__(**kwargs)

    def call(self, inputs):

        rgb_to_ycbcr_kernel = tf.constant([[0.299, 0.587, 0.114],
                                           [-0.1687, -0.3313, 0.5],
                                           [0.5, -0.4187, -0.0813]])
        offset = tf.constant([0, 128/255, 128/255], dtype=tf.float32)
        ycbcr = tf.tensordot(inputs, rgb_to_ycbcr_kernel, axes=[[3], [1]]) + offset

        return ycbcr

    def get_config(self):
        config = super(RGBtoYCbCr, self).get_config()
        return config

In [ ]:
class FuzzyPooling(Layer):
    def __init__(self, pool_size=(2, 2), strides=None, padding='VALID', fuzzy_k=2, **kwargs):
        super(FuzzyPooling, self).__init__(**kwargs)
        self.pool_size = pool_size
        self.strides = strides if strides is not None else pool_size
        self.padding = padding.upper()
        self.fuzzy_k = fuzzy_k

    def call(self, inputs):

        patches = tf.image.extract_patches(
            images=inputs,
            sizes=[1, self.pool_size[0], self.pool_size[1], 1],
            strides=[1, self.strides[0], self.strides[1], 1],
            rates=[1, 1, 1, 1],
            padding=self.padding
        )

        batch_size = tf.shape(inputs)[0]
        new_height = tf.shape(patches)[1]
        new_width = tf.shape(patches)[2]
        channels = inputs.shape[-1]
        patch_dim = self.pool_size[0] * self.pool_size[1]

        patches = tf.reshape(patches, [batch_size, new_height, new_width, channels, patch_dim])


        max_vals = tf.reduce_max(patches, axis=-1, keepdims=True)
        min_vals = tf.reduce_min(patches, axis=-1, keepdims=True)
        denom = max_vals - min_vals + 1e-6

        membership = 1 - tf.abs(patches - max_vals) / denom
        membership = tf.pow(membership, self.fuzzy_k)

        numerator = tf.reduce_sum(patches * membership, axis=-1)
        denominator = tf.reduce_sum(membership, axis=-1) + 1e-6
        fuzzy_pool = numerator / denominator

        return fuzzy_pool

    def compute_output_shape(self, input_shape):

        if self.padding == 'VALID':
            out_height = (input_shape[1] - self.pool_size[0]) // self.strides[0] + 1
            out_width = (input_shape[2] - self.pool_size[1]) // self.strides[1] + 1
        elif self.padding == 'SAME':
            out_height = (input_shape[1] + self.strides[0] - 1) // self.strides[0]
            out_width = (input_shape[2] + self.strides[1] - 1) // self.strides[1]
        else:
            raise ValueError(f"Invalid padding type: {self.padding}")

        return (input_shape[0], out_height, out_width, input_shape[3])

    def get_config(self):

        config = super(FuzzyPooling, self).get_config()
        config.update({
            'pool_size': self.pool_size,
            'strides': self.strides,
            'padding': self.padding,
            'fuzzy_k': self.fuzzy_k,
        })
        return config

In [ ]:
class GlobalFuzzyPooling2D(Layer):
    def __init__(self, fuzzy_k=2, **kwargs):
        super(GlobalFuzzyPooling2D, self).__init__(**kwargs)
        self.fuzzy_k = fuzzy_k

    def call(self, inputs):

        batch_size = tf.shape(inputs)[0]
        height = tf.shape(inputs)[1]
        width = tf.shape(inputs)[2]
        channels = inputs.shape[3]

        inputs_flat = tf.reshape(inputs, [batch_size, height * width, channels])

        max_vals = tf.reduce_max(inputs_flat, axis=1, keepdims=True)
        min_vals = tf.reduce_min(inputs_flat, axis=1, keepdims=True)
        denom = max_vals - min_vals + 1e-6

        membership = 1 - tf.abs(inputs_flat - max_vals) / denom
        membership = tf.pow(membership, self.fuzzy_k)

        numerator = tf.reduce_sum(inputs_flat * membership, axis=1)
        denominator = tf.reduce_sum(membership, axis=1) + 1e-6
        fuzzy_global_pool = numerator / denominator

        return fuzzy_global_pool

    def compute_output_shape(self, input_shape):

        return (input_shape[0], input_shape[3])

    def get_config(self):

        config = super(GlobalFuzzyPooling2D, self).get_config()
        config.update({
            'fuzzy_k': self.fuzzy_k,
        })
        return config

In [ ]:
def residual_block(x, filters, stride=1):

    shortcut = x

    x = Conv2D(filters, kernel_size=(3, 3), strides=stride, padding="same")(x)
    x = Dropout(VALUE_DT)(x, training=True)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv2D(filters, kernel_size=(3, 3), strides=1, padding="same")(x)
    x = Dropout(VALUE_DT)(x, training=True)
    x = BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, kernel_size=(1, 1), strides=stride, padding="same")(shortcut)
        shortcut = Dropout(VALUE_DT)(shortcut, training=True)
        shortcut = BatchNormalization()(shortcut)

    x = Add()([x, shortcut])
    x = ReLU()(x)

    return x

In [ ]:
def backbone_resnet18(x):
    x = Conv2D(64, kernel_size=(7, 7), strides=2, padding="same")(x)
    x = Dropout(VALUE_DT)(x, training=True)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = FuzzyPooling(pool_size=(3, 3), strides=(2, 2), padding='SAME')(x)

    x = residual_block(x, 64, stride=1)
    x = residual_block(x, 64, stride=1)

    x = residual_block(x, 128, stride=2)
    x = residual_block(x, 128, stride=1)

    x = residual_block(x, 256, stride=2)
    x = residual_block(x, 256, stride=1)

    x = residual_block(x, 512, stride=2)
    x = residual_block(x, 512, stride=1)

    return x

In [ ]:
def phd_resnet18(input_shape=INPUT_SHAPE):
    inputs = Input(shape=input_shape)

    hsv_image = RGBtoHSV()(inputs)

    ycbcr_image = RGBtoYCbCr()(inputs)

    concatenated_inputs = Concatenate()([inputs, hsv_image, ycbcr_image])

    paper_branch = backbone_resnet18(concatenated_inputs)
    replay_branch = backbone_resnet18(concatenated_inputs)
    mask_branch = backbone_resnet18(concatenated_inputs)

    liveness_branch = backbone_resnet18(concatenated_inputs)


    x = GlobalFuzzyPooling2D()(paper_branch)

    paper_output = Dense(2, activation='softmax', name='paper_output')(x)

    x = GlobalFuzzyPooling2D()(replay_branch)
    replay_output = Dense(2, activation='softmax', name='replay_output')(x)

    x = GlobalFuzzyPooling2D()(mask_branch)
    mask_output = Dense(2, activation='softmax', name='mask_output')(x)

    x = GlobalFuzzyPooling2D()(liveness_branch)
    liveness_output = Dense(2, activation='softmax', name='liveness_output')(x)

    concatenated_spoofs = Multiply()([paper_branch, replay_branch, mask_branch])

    x = Conv2D(512, kernel_size=(3, 3), padding="same")(concatenated_spoofs)
    x = Dropout(VALUE_DT)(x, training=True)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    concatenated_liveness = Concatenate()([x, liveness_branch])

    x = Conv2D(512, kernel_size=(3, 3), padding="same")(concatenated_liveness)
    x = Dropout(VALUE_DT)(x, training=True)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = GlobalFuzzyPooling2D()(x)
    liveness_final_output = Dense(2, activation='softmax', name='liveness_final_output')(x)

    model = tf.keras.models.Model(inputs, [paper_output, mask_output, replay_output, liveness_output, liveness_final_output])
    return model


#### Load model

In [ ]:
!cp "/content/gdrive/MyDrive/PhD_Models/saved_model/ResNet-18-MCD-FP_Final_MSU.keras" .

In [ ]:
model = tf.keras.models.load_model('ResNet-18-MCD-FP_Final_MSU.keras', custom_objects={
                                        'RGBtoHSV': RGBtoHSV,
                                       'RGBtoYCbCr': RGBtoYCbCr,
                                       'FuzzyPooling': FuzzyPooling,
                                       'GlobalFuzzyPooling2D': GlobalFuzzyPooling2D,
})
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 256, 256, 3)]        0         []                            
                                                                                                  
 rg_bto_hsv (RGBtoHSV)       (None, 256, 256, 3)          0         ['input_1[0][0]']             
                                                                                                  
 rg_bto_y_cb_cr (RGBtoYCbCr  (None, 256, 256, 3)          0         ['input_1[0][0]']             
 )                                                                                                
                                                                                                  
 concatenate (Concatenate)   (None, 256, 256, 9)          0         ['input_1[0][0]',         

## Load Data

In [ ]:
!cp -r "/content/gdrive/MyDrive/PhD/datasets/CelebASpoof/README/metas/" .
!cp "/content/gdrive/MyDrive/PhD/datasets/CelebASpoof/face/CelebA_Spoof_Faces.zip" .
!unzip -qq CelebA_Spoof_Faces.zip

In [ ]:
np.random.seed(42)

import json

df_train_json = pd.read_json('metas/intra_test/train_label.json').T
df_test_json = pd.read_json('metas/intra_test/test_label.json').T

select_columns = ['full_path_iqa', 'data_type', 'person', 'environment',
                  'illumination_condition', 'spoof_type', 'label', 'label_value']
list_df = []
for i in [df_train_json, df_test_json]:
    df_tmp = i.copy()
    df_tmp['full_path_iqa'] = df_tmp.index
    df_tmp['data_type'] = df_tmp.full_path_iqa.apply(lambda x: x.split('/')[1])
    df_tmp['person'] = df_tmp.full_path_iqa.apply(lambda x: x.split('/')[2])
    df_tmp['environment'] = df_tmp[42]
    df_tmp['illumination_condition'] = df_tmp[41]
    df_tmp['spoof_type'] = df_tmp[40]
    df_tmp['label'] = df_tmp.full_path_iqa.apply(lambda x: x.split('/')[-2])
    df_tmp['label_value'] = df_tmp[43]
    df_tmp = df_tmp[select_columns]
    list_df.append(df_tmp)

df = pd.concat(list_df).reset_index(drop=True)
df.full_path_iqa = df.full_path_iqa.apply(lambda x: x.replace('.png', '.jpg').replace('.jpeg', '.jpg'))
df.head(3)

,full_path_iqa,data_type,person,environment,illumination_condition,spoof_type,label,label_value
0,Data/train/2623/live/000000.jpg,train,2623,0,0,0,live,0
1,Data/train/5489/spoof/000001.jpg,train,5489,1,1,6,spoof,1
2,Data/train/7149/spoof/000002.jpg,train,7149,1,1,7,spoof,1


In [ ]:
def create_list(value, classes):
    cat_list = np.zeros(len(classes))
    cat_list[classes.index(value)] = 1
    return cat_list

In [ ]:
df['distortion_type'] = df.full_path_iqa.apply(lambda x: x.split('/')[2])

df['paper_label'] = df.spoof_type.apply(lambda x: 0 if x in [1, 2, 3] else 1).apply(create_list, args=([0, 1],))
df['mask_label'] = df.spoof_type.apply(lambda x: 0 if x in [4, 5, 6, 10] else 1).apply(create_list, args=([0, 1],))
df['replay_label'] = df.spoof_type.apply(lambda x: 0 if x in [7, 8, 9] else 1).apply(create_list, args=([0, 1],))

df['liveness_label'] = df.label.apply(lambda x: 0 if x=='spoof' else 1).apply(create_list, args=([0, 1],))
df['liveness_final_label'] = df.liveness_label

df.head(3)

,full_path_iqa,data_type,person,environment,illumination_condition,spoof_type,label,label_value,distortion_type,paper_label,mask_label,replay_label,liveness_label,liveness_final_label
0,Data/train/2623/live/000000.jpg,train,2623,0,0,0,live,0,2623,"[0.0, 1.0]","[0.0, 1.0]","[0.0, 1.0]","[0.0, 1.0]","[0.0, 1.0]"
1,Data/train/5489/spoof/000001.jpg,train,5489,1,1,6,spoof,1,5489,"[0.0, 1.0]","[1.0, 0.0]","[0.0, 1.0]","[1.0, 0.0]","[1.0, 0.0]"
2,Data/train/7149/spoof/000002.jpg,train,7149,1,1,7,spoof,1,7149,"[0.0, 1.0]","[0.0, 1.0]","[1.0, 0.0]","[1.0, 0.0]","[1.0, 0.0]"


In [ ]:
df.shape

(561575, 14)

In [ ]:
train = df[df.data_type=='train']
val = df[df.data_type=='test']

TOTAL_TRAIN = train.shape[0]
TOTAL_VAL = val.shape[0]

print(f'Total Training Samples: {TOTAL_TRAIN}')
print(f'Total Valid Samples: {TOTAL_VAL}')

Total Training Samples: 494405
Total Valid Samples: 67170


## Prepare Data

In [ ]:
val_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
val_generator = val_datagen.flow_from_dataframe(
    val,
    PATH_DIR,
    x_col='full_path_iqa',
    y_col=['paper_label', 'mask_label', 'replay_label', 'liveness_label', 'liveness_final_label'],
    target_size=IMG_SIZE,
    class_mode='multi_output',
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 67169 validated image filenames.


## Predict

In [ ]:
import tqdm
np.random.seed(42)
tf.random.set_seed(42)
no_times=25
for i in tqdm.tqdm(range(no_times), total=no_times):
    predict = np.array(model.predict(val_generator, verbose=1))[:,:,1:]

    if i==0:
        predict_test_array = predict.copy()
    else:
        predict_test_array = np.concatenate([predict_test_array, predict], axis=-1)

  0%|          | 0/25 [00:00<?, ?it/s]

263/263 [==============================] - 169s 598ms/step


  4%|▍         | 1/25 [02:50<1:08:12, 170.53s/it]

263/263 [==============================] - 156s 593ms/step


  8%|▊         | 2/25 [05:27<1:02:22, 162.70s/it]

...

 92%|█████████▏| 23/25 [1:00:28<05:14, 157.15s/it]

263/263 [==============================] - 156s 593ms/step


 96%|█████████▌| 24/25 [1:03:05<02:37, 157.16s/it]

263/263 [==============================] - 156s 593ms/step


100%|██████████| 25/25 [1:05:43<00:00, 157.73s/it]


In [ ]:
loaded_images = set()
for batch in val_generator:
    filenames = val_generator.filenames
    for filename in filenames:
        loaded_images.add(filename)
    break

dataframe_images = set(val['full_path_iqa'])

not_loaded_images = dataframe_images - loaded_images
print(not_loaded_images)

not_loaded_images_list = list(not_loaded_images)
not_loaded_images_list

{'Data/test/8963/live/516586.jpg'}


['Data/test/8963/live/516586.jpg']

In [ ]:
index = (val[val.full_path_iqa==not_loaded_images_list[0]].index-val.index[0])[0]
index

22181

In [ ]:
val['paper_pred'] = predict_test_array[0].tolist()[:index]+[None]+predict_test_array[0].tolist()[index:]#predict_test_array[0].tolist()
val['mask_pred'] = predict_test_array[1].tolist()[:index]+[None]+predict_test_array[1].tolist()[index:]#predict_test_array[1].tolist()
val['replay_pred'] = predict_test_array[2].tolist()[:index]+[None]+predict_test_array[2].tolist()[index:]#predict_test_array[2].tolist()
val['liveness_pred'] = predict_test_array[3].tolist()[:index]+[None]+predict_test_array[3].tolist()[index:]#predict_test_array[3].tolist()
val['liveness_final_pred'] = predict_test_array[4].tolist()[:index]+[None]+predict_test_array[4].tolist()[index:]#predict_test_array[4].tolist()
val.to_csv(f'/content/gdrive/MyDrive/csv_results/PhD4_protCrossDtSt_trMSU_tsCeAS.csv', index=False)
val.head(3)

<ipython-input-39-7a36a565b61c>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val['paper_pred'] = predict_test_array[0].tolist()[:index]+[None]+predict_test_array[0].tolist()[index:]#predict_test_array[0].tolist()
<ipython-input-39-7a36a565b61c>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val['mask_pred'] = predict_test_array[1].tolist()[:index]+[None]+predict_test_array[1].tolist()[index:]#predict_test_array[1].tolist()
<ipython-input-39-7a36a565b61c>:4: SettingWithCopyWarning: 
A value is trying t

,full_path_iqa,data_type,person,environment,illumination_condition,spoof_type,label,label_value,distortion_type,paper_label,mask_label,replay_label,liveness_label,liveness_final_label,paper_pred,mask_pred,replay_pred,liveness_pred,liveness_final_pred
494405,Data/test/6964/spoof/494405.jpg,test,6964,2,2,4,spoof,1,6964,"[0.0, 1.0]","[1.0, 0.0]","[0.0, 1.0]","[1.0, 0.0]","[1.0, 0.0]","[0.9999964237213135, 0.9999454021453857, 0.999...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0.00011705309589160606, 0.0004644007422029972...","[0.0001722625020192936, 2.4745409973547794e-05...","[0.0009485941845923662, 0.0001919322821777314,..."
494406,Data/test/9596/spoof/494406.jpg,test,9596,2,2,9,spoof,1,9596,"[0.0, 1.0]","[0.0, 1.0]","[1.0, 0.0]","[1.0, 0.0]","[1.0, 0.0]","[0.9999998807907104, 0.9999997615814209, 0.999...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0.27660587430000305, 0.3216278851032257, 0.72...","[0.9998630285263062, 0.9996538162231445, 0.999...","[0.9964207410812378, 0.9963621497154236, 0.998..."
494407,Data/test/9014/spoof/494407.jpg,test,9014,2,1,8,spoof,1,9014,"[0.0, 1.0]","[0.0, 1.0]","[1.0, 0.0]","[1.0, 0.0]","[1.0, 0.0]","[6.76473321803428e-10, 2.6651494344775983e-09,...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0.9999998807907104, 0.9999998807907104, 0.999...","[4.2017928025828155e-10, 1.6640877120721598e-0...","[2.0046890037740683e-13, 9.259515897086512e-13..."


END